In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

In [2]:
def discover_raw_data_dir():
    roots = []
    cwd = Path.cwd().resolve()
    roots.append(cwd)
    roots.extend(list(cwd.parents)[:4])
    roots.append(Path.home() / "Desktop")
    roots.append(Path.home())

    checked = []
    for root in roots:
        for candidate in [root / "ma-data" / "ma", root / "ma" / "ma", root / "ma"]:
            if candidate in checked:
                continue
            checked.append(candidate)
            if not candidate.exists():
                continue
            enroll_dir = candidate / "enrollment" / "Extracted Data"
            sa_dir = candidate / "service-area" / "Extracted Data"
            if enroll_dir.exists() and sa_dir.exists():
                return candidate

    for root in roots:
        if not root.exists():
            continue
        try:
            hit = next(root.rglob("CPSC_Contract_Info_2018_01.csv"))
            return hit.parents[2]
        except StopIteration:
            pass

    raise FileNotFoundError(
        "Could not find raw Medicare Advantage data. Need folders enrollment/Extracted Data and service-area/Extracted Data."
    )


DATA = discover_raw_data_dir()
ENROLL_DIR = DATA / "enrollment" / "Extracted Data"
SA_DIR = DATA / "service-area" / "Extracted Data"
OUT_DIR = Path.cwd() / "hw1_outputs"
OUT_DIR.mkdir(exist_ok=True)

print("Raw data directory:", DATA)
print("Enrollment files:", ENROLL_DIR)
print("Service area files:", SA_DIR)
print("Output directory:", OUT_DIR)

Raw data directory: /home/wfliu/econ470/a0/work/ma-data/ma
Enrollment files: /home/wfliu/econ470/a0/work/ma-data/ma/enrollment/Extracted Data
Service area files: /home/wfliu/econ470/a0/work/ma-data/ma/service-area/Extracted Data
Output directory: /home/wfliu/econ470/a0/work/hw1_outputs


In [3]:
YEAR = 2018
MONTHS = [f"{m:02d}" for m in range(1, 13)]

CONTRACT_COLUMNS = [
    "contractid", "planid", "org_type", "plan_type", "partd", "snp", "eghp",
    "org_name", "org_marketing_name", "plan_name", "parent_org", "contract_date",
]
CONTRACT_DTYPES = {
    "contractid": "string", "planid": "string", "org_type": "string", "plan_type": "string",
    "partd": "string", "snp": "string", "eghp": "string", "org_name": "string",
    "org_marketing_name": "string", "plan_name": "string", "parent_org": "string", "contract_date": "string",
}

ENROLL_COLUMNS = ["contractid", "planid", "ssa", "fips", "state", "county", "enrollment"]
ENROLL_DTYPES = {
    "contractid": "string", "planid": "string", "ssa": "string", "fips": "string",
    "state": "string", "county": "string", "enrollment": "string",
}

SA_COLUMNS = [
    "contractid", "org_name", "org_type", "plan_type", "partial", "eghp",
    "ssa", "fips", "county", "state", "notes",
]
SA_DTYPES = {
    "contractid": "string", "org_name": "string", "org_type": "string", "plan_type": "string",
    "partial": "string", "eghp": "string", "ssa": "string", "fips": "string",
    "county": "string", "state": "string", "notes": "string",
}

In [4]:
def read_csv_fallback(path, **kwargs):
    try:
        return pd.read_csv(path, encoding="utf-8", **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin1", **kwargs)


def num_planid(series):
    return pd.to_numeric(series.astype(str).str.extract(r"(\d+)")[0], errors="coerce")


def num_fips(series):
    return pd.to_numeric(series.astype(str).str.replace(".0", "", regex=False).str.extract(r"(\d+)")[0], errors="coerce")


def num_enrollment(series):
    return pd.to_numeric(series.astype(str).str.replace(",", "", regex=False), errors="coerce")


def yes_no_upper(series):
    return series.astype(str).str.strip().str.upper()


def fill_downup(df, cols):
    out = df.copy()
    out[cols] = out[cols].ffill().bfill()
    return out

In [5]:
def read_contract_month(year, month):
    path = ENROLL_DIR / f"CPSC_Contract_Info_{year}_{month}.csv"
    df = read_csv_fallback(
        path,
        skiprows=1,
        header=None,
        names=CONTRACT_COLUMNS,
        dtype=CONTRACT_DTYPES,
        low_memory=False,
    )
    df["planid"] = num_planid(df["planid"])
    df["contractid"] = df["contractid"].astype(str).str.strip().str.upper()
    return df.drop_duplicates(subset=["contractid", "planid"], keep="first")


def read_enrollment_month(year, month):
    path = ENROLL_DIR / f"CPSC_Enrollment_Info_{year}_{month}.csv"
    df = read_csv_fallback(
        path,
        skiprows=1,
        header=None,
        names=ENROLL_COLUMNS,
        dtype=ENROLL_DTYPES,
        na_values=["*"],
        low_memory=False,
    )
    df["planid"] = num_planid(df["planid"])
    df["contractid"] = df["contractid"].astype(str).str.strip().str.upper()
    df["fips"] = num_fips(df["fips"])
    df["enrollment"] = num_enrollment(df["enrollment"])
    return df


def load_enrollment_month(year, month):
    contract = read_contract_month(year, month)
    enroll = read_enrollment_month(year, month)
    return contract.merge(enroll, on=["contractid", "planid"], how="left").assign(month=int(month), year=year)


monthly_2018 = pd.concat([load_enrollment_month(YEAR, month) for month in MONTHS], ignore_index=True)
monthly_2018.head()

,contractid,planid,org_type,plan_type,partd,snp,eghp,org_name,org_marketing_name,plan_name,parent_org,contract_date,ssa,fips,state,county,enrollment,month,year
0,90091,NaN,HCPP - 1833 Cost,HCPP - 1833 Cost,No,No,No,UNITED MINE WORKERS OF AMERICA HLTH & RETIREMENT,United Mine Workers of America Health & Retire...,<NA>,UMWA Health and Retirement Funds,02/01/1974 0:00:00,<NA>,NaN,<NA>,<NA>,NaN,1,2018
1,E0654,801.0,Employer/Union Only Direct Contract PDP,Employer/Union Only Direct Contract PDP,Yes,No,Yes,IBT VOLUNTARY EMPLOYEE BENEFITS TRUST,TEAMStar Medicare Part D Prescription Drug Pro...,IBT Voluntary Employee Benefits Trust (Employe...,IBT Voluntary Employee Benefits Trust,01/01/2007 0:00:00,02198,NaN,<NA>,<NA>,NaN,1,2018
2,E0654,801.0,Employer/Union Only Direct Contract PDP,Employer/Union Only Direct Contract PDP,Yes,No,Yes,IBT VOLUNTARY EMPLOYEE BENEFITS TRUST,TEAMStar Medicare Part D Prescription Drug Pro...,IBT Voluntary Employee Benefits Trust (Employe...,IBT Voluntary Employee Benefits Trust,01/01/2007 0:00:00,55830,NaN,<NA>,<NA>,NaN,1,2018
3,E0654,801.0,Employer/Union Only Direct Contract PDP,Employer/Union Only Direct Contract PDP,Yes,No,Yes,IBT VOLUNTARY EMPLOYEE BENEFITS TRUST,TEAMStar Medicare Part D Prescription Drug Pro...,IBT Voluntary Employee Benefits Trust (Employe...,IBT Voluntary Employee Benefits Trust,01/01/2007 0:00:00,02275,NaN,<NA>,<NA>,NaN,1,2018
4,E0654,801.0,Employer/Union Only Direct Contract PDP,Employer/Union Only Direct Contract PDP,Yes,No,Yes,IBT VOLUNTARY EMPLOYEE BENEFITS TRUST,TEAMStar Medicare Part D Prescription Drug Pro...,IBT Voluntary Employee Benefits Trust (Employe...,IBT Voluntary Employee Benefits Trust,01/01/2007 0:00:00,01000,1001.0,AL,Autauga,NaN,1,2018


In [6]:
monthly_2018 = monthly_2018.sort_values(["contractid", "planid", "state", "county", "month"], kind="mergesort").copy()

monthly_2018["fips"] = (
    monthly_2018.groupby(["state", "county"], dropna=False)["fips"]
    .transform(lambda s: s.ffill().bfill())
)

plan_fill_cols = ["plan_type", "partd", "snp", "eghp", "plan_name"]
monthly_2018[plan_fill_cols] = (
    monthly_2018.groupby(["contractid", "planid"], dropna=False)[plan_fill_cols]
    .transform(lambda df: df.ffill().bfill())
)

contract_fill_cols = ["org_type", "org_name", "org_marketing_name", "parent_org"]
monthly_2018[contract_fill_cols] = (
    monthly_2018.groupby(["contractid"], dropna=False)[contract_fill_cols]
    .transform(lambda df: df.ffill().bfill())
)

monthly_2018 = monthly_2018[[
    "contractid", "planid", "fips", "year", "month", "state", "county",
    "enrollment", "org_type", "plan_type", "partd", "snp", "eghp",
    "org_name", "org_marketing_name", "plan_name", "parent_org", "contract_date",
]].copy()

print("monthly_2018 rows:", len(monthly_2018))
print("monthly_2018 memory (MB):", round(monthly_2018.memory_usage(deep=True).sum() / 1_000_000, 2))

def summarize_plan_county_year(g):
    vals = g["enrollment"].dropna()
    return pd.Series({
        "n_nonmiss": g["enrollment"].notna().sum(),
        "avg_enrollment": vals.mean() if len(vals) else np.nan,
        "sd_enrollment": vals.std(ddof=1) if len(vals) > 1 else np.nan,
        "min_enrollment": vals.min() if len(vals) else np.nan,
        "max_enrollment": vals.max() if len(vals) else np.nan,
        "first_enrollment": vals.iloc[0] if len(vals) else np.nan,
        "last_enrollment": vals.iloc[-1] if len(vals) else np.nan,
        "state": g["state"].iloc[-1],
        "county": g["county"].iloc[-1],
        "org_type": g["org_type"].iloc[-1],
        "plan_type": g["plan_type"].iloc[-1],
        "partd": g["partd"].iloc[-1],
        "snp": g["snp"].iloc[-1],
        "eghp": g["eghp"].iloc[-1],
        "org_name": g["org_name"].iloc[-1],
        "org_marketing_name": g["org_marketing_name"].iloc[-1],
        "plan_name": g["plan_name"].iloc[-1],
        "parent_org": g["parent_org"].iloc[-1],
        "contract_date": g["contract_date"].iloc[-1],
    })


plan_county_year_2018 = (
    monthly_2018
    .groupby(["contractid", "planid", "fips", "year"], dropna=False)
    .apply(summarize_plan_county_year)
    .reset_index()
)

plan_county_year_2018.head()

monthly_2018 rows: 27710394
monthly_2018 memory (MB): 24269.41


/tmp/ipykernel_3563704/725544930.py:57: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_plan_county_year)


,contractid,planid,fips,year,n_nonmiss,avg_enrollment,sd_enrollment,min_enrollment,max_enrollment,first_enrollment,last_enrollment,state,county,org_type,plan_type,partd,snp,eghp,org_name,org_marketing_name,plan_name,parent_org,contract_date
0,90091,NaN,NaN,2018,0,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,HCPP - 1833 Cost,HCPP - 1833 Cost,No,No,No,UNITED MINE WORKERS OF AMERICA HLTH & RETIREMENT,United Mine Workers of America Health & Retire...,<NA>,UMWA Health and Retirement Funds,02/01/1974 0:00:00
1,E0654,801.0,1001.0,2018,0,NaN,NaN,NaN,NaN,NaN,NaN,AL,Autauga,Employer/Union Only Direct Contract PDP,Employer/Union Only Direct Contract PDP,Yes,No,Yes,IBT VOLUNTARY EMPLOYEE BENEFITS TRUST,TEAMStar Medicare Part D Prescription Drug Pro...,IBT Voluntary Employee Benefits Trust (Employe...,IBT Voluntary Employee Benefits Trust,01/01/2007 0:00:00
2,E0654,801.0,1003.0,2018,12,13.833333,1.029857,13.0,15.0,13.0,13.0,AL,Baldwin,Employer/Union Only Direct Contract PDP,Employer/Union Only Direct Contract PDP,Yes,No,Yes,IBT VOLUNTARY EMPLOYEE BENEFITS TRUST,TEAMStar Medicare Part D Prescription Drug Pro...,IBT Voluntary Employee Benefits Trust (Employe...,IBT Voluntary Employee Benefits Trust,01/01/2007 0:00:00
3,E0654,801.0,1005.0,2018,0,NaN,NaN,NaN,NaN,NaN,NaN,AL,Barbour,Employer/Union Only Direct Contract PDP,Employer/Union Only Direct Contract PDP,Yes,No,Yes,IBT VOLUNTARY EMPLOYEE BENEFITS TRUST,TEAMStar Medicare Part D Prescription Drug Pro...,IBT Voluntary Employee Benefits Trust (Employe...,IBT Voluntary Employee Benefits Trust,01/01/2007 0:00:00
4,E0654,801.0,1007.0,2018,0,NaN,NaN,NaN,NaN,NaN,NaN,AL,Bibb,Employer/Union Only Direct Contract PDP,Employer/Union Only Direct Contract PDP,Yes,No,Yes,IBT VOLUNTARY EMPLOYEE BENEFITS TRUST,TEAMStar Medicare Part D Prescription Drug Pro...,IBT Voluntary Employee Benefits Trust (Employe...,IBT Voluntary Employee Benefits Trust,01/01/2007 0:00:00


In [8]:
def read_service_area_month(year, month):
    path = SA_DIR / f"MA_Cnty_SA_{year}_{month}.csv"
    df = read_csv_fallback(
        path,
        skiprows=1,
        header=None,
        names=SA_COLUMNS,
        dtype=SA_DTYPES,
        na_values=["*"],
        low_memory=False,
    )
    df["contractid"] = df["contractid"].astype(str).str.strip().str.upper()
    df["fips"] = num_fips(df["fips"])
    df["month"] = int(month)
    df["year"] = year
    return df


service_2018 = pd.concat([read_service_area_month(YEAR, month) for month in MONTHS], ignore_index=True)
service_2018 = service_2018.sort_values(["contractid", "fips", "state", "county", "month"], kind="mergesort")
service_2018["fips"] = service_2018.groupby(["state", "county"], dropna=False)["fips"].transform(lambda s: s.ffill().bfill())
service_2018[["plan_type", "partial", "eghp", "org_type", "org_name"]] = (
    service_2018.groupby(["contractid"], dropna=False)[["plan_type", "partial", "eghp", "org_type", "org_name"]]
    .transform(lambda df: df.ffill().bfill())
)

service_area_2018 = (
    service_2018
    .groupby(["contractid", "fips", "year"], dropna=False, as_index=False)
    .agg(
        state=("state", "last"),
        county=("county", "last"),
        org_name=("org_name", "last"),
        org_type=("org_type", "last"),
        plan_type_sa=("plan_type", "last"),
        partial=("partial", "last"),
        eghp_sa=("eghp", "last"),
        ssa=("ssa", "last"),
        notes=("notes", "last"),
    )
)

service_area_2018.head()

,contractid,fips,year,state,county,org_name,org_type,plan_type_sa,partial,eghp_sa,ssa,notes
0,90091,NaN,2018,<NA>,<NA>,UNITED MINE WORKERS OF AMERICA HLTH & RETIREMENT,HCPP - 1833 Cost,HCPP - 1833 Cost,<NA>,<NA>,<NA>,"Covers the entire US, all States and Counties"
1,H0022,39023.0,2018,OH,Clark,"BUCKEYE COMMUNITY HEALTH PLAN, INC.",Demo,Medicare-Medicaid Plan HMO/HMOPOS,<NA>,<NA>,36110,<NA>
2,H0022,39035.0,2018,OH,Cuyahoga,"BUCKEYE COMMUNITY HEALTH PLAN, INC.",Demo,Medicare-Medicaid Plan HMO/HMOPOS,<NA>,<NA>,36170,<NA>
3,H0022,39051.0,2018,OH,Fulton,"BUCKEYE COMMUNITY HEALTH PLAN, INC.",Demo,Medicare-Medicaid Plan HMO/HMOPOS,<NA>,<NA>,36260,<NA>
4,H0022,39055.0,2018,OH,Geauga,"BUCKEYE COMMUNITY HEALTH PLAN, INC.",Demo,Medicare-Medicaid Plan HMO/HMOPOS,<NA>,<NA>,36280,<NA>


In [9]:
approved_2018 = plan_county_year_2018.merge(
    service_area_2018,
    on=["contractid", "fips", "year"],
    how="inner",
)

approved_2018["planid"] = pd.to_numeric(approved_2018["planid"], errors="coerce")
approved_2018["plan_type"] = approved_2018["plan_type"].fillna(approved_2018["plan_type_sa"])

plan_county_year_2018.to_csv(OUT_DIR / "plan_county_year_2018.csv", index=False)
service_area_2018.to_csv(OUT_DIR / "service_area_2018.csv", index=False)
approved_2018.to_csv(OUT_DIR / "approved_plan_county_year_2018.csv", index=False)

print("Collapsed plan-county-year rows:", len(plan_county_year_2018))
print("Approved rows after inner merge:", len(approved_2018))

Collapsed plan-county-year rows: 2475118
Approved rows after inner merge: 1366542


In [ ]:
#Question 1

In [10]:
table1 = (
    plan_county_year_2018
    .drop_duplicates(subset=["contractid", "planid"])
    .groupby("plan_type", dropna=False)
    .size()
    .reset_index(name="2018")
    .sort_values(["2018", "plan_type"], ascending=[False, True])
)

table1.to_csv(OUT_DIR / "table1_plan_count_by_type_2018.csv", index=False)
table1

,plan_type,2018
3,HMO/HMOPOS,2678
6,Medicare Prescription Drug Plan,1011
4,Local PPO,966
8,National PACE,258
10,Regional PPO,109
0,1876 Cost,101
7,Medicare-Medicaid Plan HMO/HMOPOS,54
9,PFFS,50
2,HCPP - 1833 Cost,9
5,MSA,5


In [ ]:
#Question 2

In [11]:
filtered_2018 = plan_county_year_2018.copy()
filtered_2018["snp_flag"] = yes_no_upper(filtered_2018["snp"])
filtered_2018["eghp_flag"] = yes_no_upper(filtered_2018["eghp"])

filtered_2018 = filtered_2018[
    (filtered_2018["snp_flag"] != "YES")
    & (filtered_2018["eghp_flag"] != "YES")
    & (filtered_2018["planid"] < 800)
].copy()

table2 = (
    filtered_2018
    .drop_duplicates(subset=["contractid", "planid"])
    .groupby("plan_type", dropna=False)
    .size()
    .reset_index(name="2018")
    .sort_values(["2018", "plan_type"], ascending=[False, True])
)

table2.to_csv(OUT_DIR / "table2_filtered_plan_count_by_type_2018.csv", index=False)
table2

,plan_type,2018
1,HMO/HMOPOS,1569
4,Medicare Prescription Drug Plan,794
2,Local PPO,569
6,National PACE,258
0,1876 Cost,89
5,Medicare-Medicaid Plan HMO/HMOPOS,54
8,Regional PPO,49
7,PFFS,46
3,MSA,3


In [ ]:
#Question 3

In [12]:
approved_filtered_2018 = approved_2018.copy()
approved_filtered_2018["snp_flag"] = yes_no_upper(approved_filtered_2018["snp"])
approved_filtered_2018["eghp_flag"] = yes_no_upper(approved_filtered_2018["eghp"])

approved_filtered_2018 = approved_filtered_2018[
    (approved_filtered_2018["snp_flag"] != "YES")
    & (approved_filtered_2018["eghp_flag"] != "YES")
    & (approved_filtered_2018["planid"] < 800)
].copy()

table3 = (
    approved_filtered_2018
    .groupby("plan_type", dropna=False)
    .agg(avg_enrollment=("avg_enrollment", "mean"), number_of_rows=("plan_type", "size"))
    .reset_index()
    .sort_values(["avg_enrollment", "plan_type"], ascending=[False, True])
)

table3["avg_enrollment"] = table3["avg_enrollment"].round(2)
table3.to_csv(OUT_DIR / "table3_avg_enrollment_by_type_approved_filtered_2018.csv", index=False)
table3

,plan_type,avg_enrollment,number_of_rows
4,Medicare-Medicaid Plan HMO/HMOPOS,989.17,418
1,HMO/HMOPOS,755.55,33236
2,Local PPO,330.62,38574
0,1876 Cost,253.10,4840
7,Regional PPO,188.79,6797
5,National PACE,144.33,902
6,PFFS,93.66,2726
3,MSA,58.13,183


In [13]:
print("Saved outputs:")
for path in sorted(OUT_DIR.glob("*.csv")):
    print("-", path.name)

Saved outputs:
- approved_plan_county_year_2018.csv
- plan_county_year_2018.csv
- service_area_2018.csv
- table1_plan_count_by_type_2018.csv
- table2_filtered_plan_count_by_type_2018.csv
- table3_avg_enrollment_by_type_approved_filtered_2018.csv
